In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "README.md").exists()
    and (path / "Data_making").is_dir()
    and (path / "Models").is_dir()
)
DATA_PATH = (
    PROJECT_ROOT
    / "Data_making"
    / "synthetic_education_dushanbe_WITH_ROUNDED.csv"
)

np.random.seed(2026)

In [2]:
# ======================================================
# РАЗМЕР
# ======================================================
N = 1525

In [3]:
# ======================================================
# РАЙОНЫ
# ======================================================
районы = ["Исмоили Сомони", "Сино", "Фирдавси", "Шохмансур"]
доли = [0.30, 0.28, 0.22, 0.20]

район_эффект = {
    "Исмоили Сомони": 0.06,
    "Сино": 0.03,
    "Фирдавси": 0.00,
    "Шохмансур": -0.04
}

df = pd.DataFrame({
    "ID_ученика": np.arange(1, N+1),
    "Класс": 11,
    "Район": np.random.choice(районы, N, p=доли)
})

сдвиг = df["Район"].map(район_эффект).values

In [4]:
# ======================================================
# ДВА ЛАТЕНТНЫХ ФАКТОРА
# ======================================================
академ = np.clip(np.random.normal(0.55, 0.15, N) + сдвиг, 0, 1)
дисциплина = np.clip(
    0.6 * академ + np.random.normal(0, 0.18, N),
    0, 1
)

In [5]:
# ======================================================
# НАБЛЮДАЕМЫЕ ПРИЗНАКИ
# ======================================================
df["Средний_балл"] = np.round(
    np.clip(3 + 1.9 * академ + np.random.normal(0, 0.15, N), 3, 5),
    3
)

df["Часы_самоподготовки_в_неделю"] = np.clip(
    np.round(5 + 20 * дисциплина + np.random.normal(0, 1.5, N)).astype(int),
    0, 30
)

df["Посещаемость_%"] = np.round(
    np.clip(70 + 25 * дисциплина + np.random.normal(0, 2.5, N), 60, 100),
    3
)

df["Уверенность_в_себе"] = np.clip(
    np.round(1 + 9*(0.7*академ + np.random.normal(0,0.08,N))),
    1, 10
)

df["Уровень_стресса_перед_экзаменом"] = np.clip(
    np.round(1 + 9*(0.8 - 0.6*академ + np.random.normal(0,0.08,N))),
    1, 10
)

df["Пропущенные_дни"] = np.clip(
    np.round(14 - 11 * дисциплина + np.random.normal(0, 2.0, N)).astype(int),
    0, 30
)

df["Тип_школы"] = np.where(академ > 0.65, "Лицей/гимназия", "Обычная школа")

In [6]:
# ======================================================
# ЦЕЛЕВАЯ ПЕРЕМЕННАЯ
# ======================================================
stress = df["Уровень_стресса_перед_экзаменом"] / 10
attendance = df["Посещаемость_%"] / 100
confidence = df["Уверенность_в_себе"] / 10

logit = (
    -3.4
    + 2.4 * (df["Средний_балл"] - 3)
    + 1.8 * (df["Часы_самоподготовки_в_неделю"] / 30)
    + 1.6 * attendance
    + 1.9 * confidence
    - 1.4 * stress
    - 1.6 * (df["Пропущенные_дни"] / 30)
    + np.random.normal(0, 0.18, N)
)

p = 1 / (1 + np.exp(-logit))

df["Рез_экзамена"] = (np.random.rand(N) < p).astype(int)

In [7]:
# ======================================================
# ДОПОЛНИТЕЛЬНЫЕ ПОКАЗАТЕЛИ КАЧЕСТВА ОБРАЗОВАНИЯ
# ======================================================
np.random.seed(2027)

avg_grade_norm = (df["Средний_балл"] - 3) / 2
attendance_norm = df["Посещаемость_%"] / 100
confidence_norm = df["Уверенность_в_себе"] / 10
stress_norm = df["Уровень_стресса_перед_экзаменом"] / 10

df["Индекс_качества_школы"] = np.round(np.clip(
    0.35 * avg_grade_norm
    + 0.25 * attendance_norm
    + 0.25 * confidence_norm
    - 0.15 * stress_norm
    + np.random.normal(0, 0.08, len(df)),
    0, 1
), 4)

df["Стабильность_преподавателей"] = np.round(np.clip(
    0.6 * df["Индекс_качества_школы"]
    + np.random.normal(0, 0.10, len(df)),
    0, 1
), 4)

df["Доступ_к_ресурсам"] = np.round(np.clip(
    0.5 * df["Индекс_качества_школы"]
    + 0.3 * avg_grade_norm
    + np.random.normal(0, 0.12, len(df)),
    0, 1
), 4)

df["Образовательная_среда"] = np.round(np.clip(
    0.4 * confidence_norm
    + 0.4 * attendance_norm
    - 0.2 * stress_norm
    + np.random.normal(0, 0.10, len(df)),
    0, 1
), 4)

In [8]:
# ======================================================
# СОХРАНЕНИЕ
# ======================================================
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(DATA_PATH, index=False)

print("Датасет сохранён с округлёнными значениями")

Датасет сохранён с округлёнными значениями


In [9]:
df.head()

,ID_ученика,Класс,Район,Средний_балл,Часы_самоподготовки_в_неделю,Посещаемость_%,Уверенность_в_себе,Уровень_стресса_перед_экзаменом,Пропущенные_дни,Тип_школы,Рез_экзамена,Индекс_качества_школы,Стабильность_преподавателей,Доступ_к_ресурсам,Образовательная_среда
0,1,11,Исмоили Сомони,3.883,15,81.091,4.0,6.0,8,Обычная школа,1,0.4000,0.2911,0.3107,0.1970
1,2,11,Сино,3.915,15,87.872,4.0,4.0,10,Обычная школа,0,0.4443,0.2559,0.4509,0.5341
2,3,11,Шохмансур,3.660,5,71.282,5.0,6.0,13,Обычная школа,0,0.2499,0.1305,0.1049,0.2666
3,4,11,Исмоили Сомони,4.401,15,84.693,5.0,5.0,10,Обычная школа,1,0.5137,0.5229,0.4238,0.4449
4,5,11,Сино,4.311,12,75.941,5.0,4.0,9,Лицей/гимназия,1,0.4473,0.4858,0.4935,0.4143


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1525 entries, 0 to 1524
Data columns (total 15 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   ID_ученика                       1525 non-null   int64  
 1   Класс                            1525 non-null   int64  
 2   Район                            1525 non-null   object 
 3   Средний_балл                     1525 non-null   float64
 4   Часы_самоподготовки_в_неделю     1525 non-null   int64  
 5   Посещаемость_%                   1525 non-null   float64
 6   Уверенность_в_себе               1525 non-null   float64
 7   Уровень_стресса_перед_экзаменом  1525 non-null   float64
 8   Пропущенные_дни                  1525 non-null   int64  
 9   Тип_школы                        1525 non-null   object 
 10  Рез_экзамена                     1525 non-null   int64  
 11  Индекс_качества_школы            1525 non-null   float64
 12  Стабильность_препода

In [11]:
df.corr(numeric_only=True)

,ID_ученика,Класс,Средний_балл,Часы_самоподготовки_в_неделю,Посещаемость_%,Уверенность_в_себе,Уровень_стресса_перед_экзаменом,Пропущенные_дни,Рез_экзамена,Индекс_качества_школы,Стабильность_преподавателей,Доступ_к_ресурсам,Образовательная_среда
ID_ученика,1.000000,NaN,-0.002275,0.017503,0.005662,0.008396,0.014438,-0.000148,0.021568,-0.011871,-0.036040,0.007225,0.009563
Класс,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Средний_балл,-0.002275,NaN,1.000000,0.363097,0.375736,0.697737,-0.659722,-0.279540,0.482962,0.757037,0.445739,0.627161,0.448299
Часы_самоподготовки_в_неделю,0.017503,NaN,0.363097,1.000000,0.827030,0.303034,-0.288556,-0.656671,0.317989,0.383004,0.220146,0.258780,0.338704
Посещаемость_%,0.005662,NaN,0.375736,0.827030,1.000000,0.297010,-0.293420,-0.641772,0.330130,0.394672,0.232423,0.257610,0.360707
Уверенность_в_себе,0.008396,NaN,0.697737,0.303034,0.297010,1.000000,-0.574394,-0.211939,0.383795,0.671523,0.391535,0.507555,0.541085
Уровень_стресса_перед_экзаменом,0.014438,NaN,-0.659722,-0.288556,-0.293420,-0.574394,1.000000,0.206538,-0.351812,-0.590697,-0.356116,-0.480118,-0.465195
Пропущенные_дни,-0.000148,NaN,-0.279540,-0.656671,-0.641772,-0.211939,0.206538,1.000000,-0.260909,-0.289918,-0.173600,-0.197332,-0.245971
Рез_экзамена,0.021568,NaN,0.482962,0.317989,0.330130,0.383795,-0.351812,-0.260909,1.000000,0.374707,0.235169,0.302331,0.283010
Индекс_качества_школы,-0.011871,NaN,0.757037,0.383004,0.394672,0.671523,-0.590697,-0.289918,0.374707,1.000000,0.601978,0.657721,0.432225


In [12]:
df.describe()

,ID_ученика,Класс,Средний_балл,Часы_самоподготовки_в_неделю,Посещаемость_%,Уверенность_в_себе,Уровень_стресса_перед_экзаменом,Пропущенные_дни,Рез_экзамена,Индекс_качества_школы,Стабильность_преподавателей,Доступ_к_ресурсам,Образовательная_среда
count,1525.000000,1525.0,1525.000000,1525.000000,1525.000000,1525.000000,1525.000000,1525.000000,1525.000000,1525.000000,1525.000000,1525.000000,1525.000000
mean,763.000000,11.0,4.078284,11.868197,78.673868,4.603279,5.127213,10.245902,0.622295,0.423039,0.255929,0.371837,0.392824
std,440.373894,0.0,0.341046,4.178632,5.506703,1.274365,1.168068,2.911060,0.484972,0.131241,0.124497,0.158407,0.128124
min,1.000000,11.0,3.062000,2.000000,64.546000,1.000000,1.000000,0.000000,0.000000,0.017700,0.000000,0.000000,0.000000
25%,382.000000,11.0,3.843000,9.000000,74.913000,4.000000,4.000000,8.000000,0.000000,0.332800,0.167500,0.265300,0.305400
50%,763.000000,11.0,4.085000,12.000000,78.684000,5.000000,5.000000,10.000000,1.000000,0.422100,0.254400,0.374400,0.393700
75%,1144.000000,11.0,4.306000,15.000000,82.305000,5.000000,6.000000,12.000000,1.000000,0.512600,0.340700,0.478300,0.476200
max,1525.000000,11.0,5.000000,26.000000,96.614000,8.000000,9.000000,19.000000,1.000000,0.809400,0.687000,0.932900,0.857500


Идентификаторы и контекст

	•	ID_ученика — уникальный идентификатор ученика
	•	Класс — класс обучения (фиксировано: 11)
	•	Район — район города Душанбе (категориальный)

⸻

Академические результаты

	•	Средний_балл — средний школьный балл ученика (3–5)
	•	Часы_самоподготовки_в_неделю — часы самостоятельной учёбы в неделю (0–30)
	•	Посещаемость_% — процент посещённых занятий (55–100)

⸻

Психология и поведение

	•	Уверенность_в_себе — субъективная уверенность ученика (1–10)
	•	Уровень_стресса_перед_экзаменом — уровень экзаменационного стресса (1–10)
	•	Пропущенные_дни — количество пропущенных учебных дней (0–35)

⸻

Характеристика школы

	•	Тип_школы — тип учебного заведения (Обычная школа / Лицей/гимназия)

⸻

Показатели качества образования (добавленные)

	•	Индекс_качества_школы — агрегированный индекс качества обучения (0–1)
	•	Стабильность_преподавателей — стабильность и опыт педсостава (0–1)
	•	Доступ_к_ресурсам — обеспеченность учебными ресурсами (0–1)
	•	Образовательная_среда — учебная атмосфера и дисциплина (0–1)

⸻

Целевая переменная

	•	Рез_экзамена — сдал ли ученик экзамен
	•	1 — сдал
	•	0 — не сдал

⸻